# 01 · Diccionario de datos — ENCC 2017

Este cuaderno recalcula el perfil automático del diccionario sobre las **2.800 filas válidas**, le agrega a mano qué preguntas usa cada variable, los valores imposibles y la decisión tomada, y guarda el resultado en `diccionario_encc_v2.csv`.

El origen, la licencia y los problemas de calidad del conjunto están en el `README.md`.

**Preguntas de análisis (versión de trabajo):**

1. **P1** · % que escucha música por dispositivo digital (`p13_3` a `p13_6`), por región.
2. **P2** · Minutos diarios de internet (`minutos_internet_total`) por grupo etario.
3. **P3** · % que no leyó ningún libro (`p42`), por nivel educativo (`p106`).
4. **P4** · Diferencia en minutos de redes (`minutos_redes_total`) entre varones y mujeres.
5. **P5** · Plataforma de música online más mencionada (`p18a`).
6. **P6** · % que fue al cine (`p63`), CABA contra el resto.
7. **P7** · Distribución de `NSEcat1` por región.
8. **P8** · Minutos diarios de videojuegos (`minutos_videojuegos_total`), menores de 25 contra el total.
9. **P9** · % que se conecta por celular todos o casi todos los días (`p75_2`), por edad.
10. **P10** · % que participó en alguna actividad presencial (`p98`, `p23`, `p69`), CABA/PBA contra el resto.

## 1. Lectura

Todo se lee como texto: así no se pierden ceros ni se confunde el `"NA"` (no aplica) con un vacío real.

In [34]:
from pathlib import Path
import numpy as np
import pandas as pd

# Sube desde donde esté el cuaderno hasta encontrar la carpeta que tiene `datos/`
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datos').is_dir())
RUTA = RAIZ / 'datos' / 'crudos' / 'encc_2017.xlsx'
assert RUTA.exists(), f'no se encuentra {RUTA.relative_to(RAIZ)}'

enc = pd.read_excel(RUTA, dtype=str, keep_default_na=False, na_values=[''])
print('crudo -> filas:', len(enc), '| id nulos:', enc['id'].isna().sum(), '| id únicos:', enc['id'].nunique())

crudo -> filas: 2802 | id nulos: 1 | id únicos: 2801


**Registro de decisiones #1:** se descartan las filas 1949 y 2739 (columnas desplazadas, `sexo` = "ALQUILADA"). Con eso la clave `id` queda única y sin nulos.

In [35]:
enc = enc[enc['sexo'] != 'ALQUILADA'].reset_index(drop=True)
assert enc['id'].notna().all(), 'hay ids nulos'
assert enc['id'].is_unique, 'el id se repite'
assert len(enc) == 2800

## 2. Perfil recalculado

Se separan dos tipos de ausencia: el texto `"NA"` (no aplica, por salto del cuestionario) y los vacíos reales.

In [36]:
dic = pd.read_csv(RAIZ / 'diccionario_encc.csv').set_index('columna')
assert list(dic.index) == list(enc.columns)

sin_na = enc.mask(enc == 'NA')
dic['no nulos']       = sin_na.notna().sum()
dic['% nulos']        = (100 * sin_na.isna().mean()).round(2)
dic['únicos']         = sin_na.nunique()
dic['no aplica (NA)'] = (enc == 'NA').sum()
dic['vacíos reales']  = enc.isna().sum()
dic['NS/NC (texto)']  = (enc == 'NS/NC').sum()

## 3. Qué pregunta usa cada variable

In [37]:
USO = {
    'P1':  ['p10', 'p13_3', 'p13_4', 'p13_5', 'p13_6', 'region'],
    'P2':  ['minutos_internet_total', 'edad'],
    'P3':  ['p42', 'p106'],
    'P4':  ['minutos_redes_total', 'sexo'],
    'P5':  ['p18a'],
    'P6':  ['p63', 'region'],
    'P7':  ['NSEcat1', 'region', 'pondera_dem'],
    'P8':  ['minutos_videojuegos_total', 'edad'],
    'P9':  ['p75_2', 'edad'],
    'P10': ['p98', 'p23', 'p69', 'region'],
}
assert not [c for cs in USO.values() for c in cs if c not in dic.index]

dic['pregunta afectada'] = ''
for p, cols in USO.items():
    for c in cols:
        dic.loc[c, 'pregunta afectada'] = (dic.loc[c, 'pregunta afectada'] + ' ' + p).strip()

## 4. Valores imposibles

Se separan de los atípicos: un valor imposible no puede ocurrir; un atípico es raro pero posible.

In [38]:
IMPOSIBLE = {'edad': 'menor de 13 (la encuesta es de 13 años y más)'}
for t in ['internet', 'redes', 'videojuegos', 'musica']:
    IMPOSIBLE[f'minutos_{t}_total'] = 'más de 1440 (un día tiene 1440 min)'

dic['valores imposibles'] = ''
for c, texto in IMPOSIBLE.items():
    dic.loc[c, 'valores imposibles'] = texto

for c in [c for c in IMPOSIBLE if c.startswith('minutos_')]:
    print(c, int((pd.to_numeric(enc[c], errors='coerce') > 1440).sum()), 'imposibles')
print('edad < 13:', int((pd.to_numeric(enc['edad'], errors='coerce') < 13).sum()))

minutos_internet_total 0 imposibles
minutos_redes_total 0 imposibles
minutos_videojuegos_total 0 imposibles
minutos_musica_total 1 imposibles
edad < 13: 0


## 5. Decisión tomada por variable

Una línea corta por variable. Las decisiones que necesitan evidencia y alternativas descartadas están en el **registro de decisiones**. `anotar` no repite un texto que ya está, así que se puede correr varias veces.

In [39]:
dic['decisión tomada'] = dic['decisión tomada'].fillna('')

def anotar(cols, texto):
    for c in cols:
        d = dic.loc[c, 'decisión tomada']
        if texto not in d:
            dic.loc[c, 'decisión tomada'] = texto if d == '' else d + ' | ' + texto

# --- Por familias de columnas
horas = [c for c in dic.index if c.startswith('horas_') and c.endswith('_total')]
anotar(horas, 'Descartada: dañada al convertir a Excel (fechas y valores > 24 h); se usa minutos_*_total, equivalente a horas x 60')
anotar(dic.index[dic['no aplica (NA)'] > 0], "'NA' = no aplica por salto del cuestionario: se conserva, no se imputa")
anotar(dic.index[dic['NS/NC (texto)'] > 0], 'NS/NC (texto) pasa a NaN: no es una respuesta real')
anotar(['p110', 'p111'], 'Texto libre: pendiente de normalizar mayúsculas y espacios para agrupar; se conserva el original')

# --- Variables de las preguntas
anotar(['region'], 'Sin cambios: 7 categorías de 400 casos, sin nulos')
anotar(['sexo'], "Sin cambios: 'ALQUILADA' venía de las 2 filas corruptas descartadas")
anotar(['edad'], 'Convertida a entero: 2800 valores numéricos, rango 13-92, sin imposibles')
anotar(['pondera_dem'], 'Convertida a número; se reporta con y sin ponderar (rango 2 a 182.969)')
anotar(['p10', 'p23', 'p63', 'p98'], 'Sin cambios: SI/NO sin nulos')
anotar(['p42'], 'Los 17 vacíos se conservan como NaN, no se imputan; se excluyen de la base de P3')
anotar(['p106'], 'Ordinal de 10 niveles (SIN ESTUDIOS a POSGRADO; confirmar el orden); sin vacíos; categorías chicas (22 y 19 casos) a agrupar para el chi²')
anotar(['NSEcat1'], 'Declarada ordinal AB>C1>C2>C3>D1>D2>E; 2 vacíos como NaN')
anotar(['minutos_musica_total'], 'Un valor de 1830 min (30,5 h/día) imposible: pasa a NaN, no se elimina la fila')
anotar(['minutos_internet_total', 'minutos_redes_total', 'minutos_videojuegos_total'],
       'Convertida a número; 0 = no usa y se conserva; 2 vacíos como NaN')

## 6. Verificación y guardado

In [40]:
sin_decision = dic[(dic['decisión tomada'] == '') & (dic['pregunta afectada'] != '')]
assert len(sin_decision) == 0, f'variables de las preguntas sin decisión: {list(sin_decision.index)}'

# Una sola decisión por variable en las que antes se duplicaban
for c in ['edad', 'p42', 'NSEcat1']:
    assert dic.loc[c, 'decisión tomada'].count(' | ') == 0, f'{c} tiene más de una decisión'

por_confirmar = list(dic.index[dic['decisión tomada'].str.contains('PROPUESTA')])
print('decisiones marcadas como PROPUESTA (confirmar):', por_confirmar)

dic.reset_index().to_csv(RAIZ / 'diccionario_encc_v2.csv', index=False, encoding='utf-8-sig')
print('guardado:', (RAIZ / 'diccionario_encc_v2.csv').relative_to(RAIZ), '|', dic.shape[0], 'variables')

decisiones marcadas como PROPUESTA (confirmar): ['p47_1d', 'p57_1otr', 'p81_1l']
guardado: diccionario_encc_v2.csv | 450 variables
